# 1. Project Title: Data Cleaning and Preprocessing Pipeline

## 2. Objective
Clean, validate, and merge the raw traffic, weather, and holiday data streams. Address physical sensor anomalies (0 K dropouts, extreme precipitation spikes), handle duplicate timestamps, create temporal features, and generate the master processed dataset: `../datasets/processed/traffic_processed.csv`.

In [1]:
import os
import pandas as pd
import numpy as np

RAW_DIR = "../datasets/raw"
PROCESSED_DIR = "../datasets/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

df_traffic = pd.read_csv(os.path.join(RAW_DIR, "traffic_raw.csv"))
df_weather = pd.read_csv(os.path.join(RAW_DIR, "weather_raw.csv"))
df_holidays = pd.read_csv(os.path.join(RAW_DIR, "holidays_raw.csv"))

print(f"Raw records ingested: Traffic ({len(df_traffic):,}), Weather ({len(df_weather):,}), Holidays ({len(df_holidays):,})")

Raw records ingested: Traffic (48,204), Weather (48,204), Holidays (48,204)


## 3. Timestamp Standardization and Parsing

In [2]:
# Convert date_time to proper datetime objects
df_traffic['date_time'] = pd.to_datetime(df_traffic['date_time'])
df_weather['date_time'] = pd.to_datetime(df_weather['date_time'])
df_holidays['date_time'] = pd.to_datetime(df_holidays['date_time'])

print(f"Traffic Date Range: {df_traffic['date_time'].min()} to {df_traffic['date_time'].max()}")

Traffic Date Range: 2012-10-02 09:00:00 to 2018-09-30 23:00:00


## 4. Anomaly and Outlier Cleansing in Meteorological Telemetry
- Temperature $0\text{ K}$ ($-273.15^\circ\text{C}$) represents sensor communication loss; replaced with median/filtered valid range.
- Rainfall spike of $9,831\text{ mm}$ (physically impossible in 1 hour) capped/cleaned.

In [3]:
# Filter invalid temperature readings (< 200 K = -73 C)
temp_outliers = (df_weather['temp'] < 200)
print(f"Sensor temperature dropouts detected: {temp_outliers.sum()}")
median_temp = df_weather.loc[~temp_outliers, 'temp'].median()
df_weather.loc[temp_outliers, 'temp'] = median_temp

# Cap extreme rain anomaly
rain_outliers = (df_weather['rain_1h'] > 500)
print(f"Extreme rain spikes detected: {rain_outliers.sum()}")
df_weather.loc[rain_outliers, 'rain_1h'] = 0.0

print("Meteorological anomalies remediated.")

Sensor temperature dropouts detected: 10
Extreme rain spikes detected: 1
Meteorological anomalies remediated.


## 5. Merging Multi-Source Data Streams

In [4]:
# Fill missing holidays with 'None'
df_holidays['holiday'] = df_holidays['holiday'].fillna('None')

# Merge traffic, weather, and holidays on date_time
df_merged = df_traffic.merge(df_weather, on='date_time', how='inner')
df_merged = df_merged.merge(df_holidays, on='date_time', how='inner')

# Drop duplicate timestamps, keeping first valid observation
df_merged = df_merged.drop_duplicates(subset=['date_time']).sort_values('date_time').reset_index(drop=True)
print(f"Clean merged shape after deduplication: {df_merged.shape}")

Clean merged shape after deduplication: (40575, 9)


## 6. Engineered Temporal Features
Constructing cyclical and calendar variables: `hour`, `day`, `day_of_week`, `month`, `year`, `is_weekend`, `is_rush_hour` (Morning: 7-9, Evening: 16-19 weekdays).

In [5]:
df_merged['hour'] = df_merged['date_time'].dt.hour
df_merged['day'] = df_merged['date_time'].dt.day
df_merged['day_of_week'] = df_merged['date_time'].dt.dayofweek
df_merged['month'] = df_merged['date_time'].dt.month
df_merged['year'] = df_merged['date_time'].dt.year

df_merged['is_weekend'] = df_merged['day_of_week'].isin([5, 6]).astype(int)
df_merged['is_rush_hour'] = (
    (df_merged['is_weekend'] == 0) & 
    (df_merged['hour'].isin([7, 8, 9, 16, 17, 18, 19]))
).astype(int)

print("Temporal feature engineering completed.")
display(df_merged[['date_time', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'traffic_volume']].head())

Temporal feature engineering completed.


,date_time,hour,day_of_week,is_weekend,is_rush_hour,traffic_volume
0,2012-10-02 09:00:00,9,1,0,1,5545
1,2012-10-02 10:00:00,10,1,0,0,4516
2,2012-10-02 11:00:00,11,1,0,0,4767
3,2012-10-02 12:00:00,12,1,0,0,5026
4,2012-10-02 13:00:00,13,1,0,0,4918


## 7. Exporting Clean Processed Dataset

In [6]:
processed_file_path = os.path.join(PROCESSED_DIR, "traffic_processed.csv")
df_merged.to_csv(processed_file_path, index=False)
print(f"Successfully exported clean processed dataset to {processed_file_path}")
print(f"Final Processed Dimensions: {df_merged.shape[0]:,} rows, {df_merged.shape[1]} columns")

Successfully exported clean processed dataset to ../datasets/processed\traffic_processed.csv
Final Processed Dimensions: 40,575 rows, 16 columns
